# Phase 2: Model Training & Benchmarking

**Project:** ML-Powered Loss Reserve Estimator  
**Objective:** Train an XGBoost regressor to predict ultimate paid losses and benchmark its accuracy against a traditional Chain-Ladder baseline.

**Why paid basis:** The CAS database records net incurred losses inclusive of reserve releases, causing incurred losses to *decrease* from lag 1 to lag 10 in ~70% of cases. Paid losses develop upward in ~74% of cases and show large development factors (e.g. medmal: 68x from lag 1 to ultimate). Both the ML model and chain-ladder baseline are therefore evaluated on a paid loss basis for a consistent, meaningful comparison.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

sys.path.append(os.path.abspath('../'))

from src.data.loader import load_all
from src.data.cleaner import clean
from src.data.features import create_ml_features, X_COLS
from src.models.baseline import ActuarialBaseline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]

df = clean(load_all())
ml_data = create_ml_features(df)
ml_data['accident_year'] = pd.to_numeric(ml_data['accident_year'])

print(f'Full ML dataset: {ml_data.shape[0]:,} rows')
print(f'Accident years: {ml_data.accident_year.min()} – {ml_data.accident_year.max()}')
print(f'Target: ultimate paid loss at lag 10')
print(f'Features: {X_COLS}')

## 1. Train / test split

We use a **chronological split** — the only valid approach for time-series data. Training on future data to predict the past would be leakage.

- **Train:** accident years 1998–2002
- **Test:** accident years 2003–2007 — held out, never seen during training

This mirrors how a reserving model would be deployed: trained on historical triangles, evaluated on more recent ones.

In [ ]:
train = ml_data[ml_data.accident_year <= 2002].copy()
test  = ml_data[ml_data.accident_year >  2002].copy()

print(f'Train: {len(train):,} rows | accident years {train.accident_year.min()}–{train.accident_year.max()}')
print(f'Test:  {len(test):,} rows  | accident years {test.accident_year.min()}–{test.accident_year.max()}')
print()

line_cols = [c for c in ml_data.columns if c.startswith('line_')]
print('Lines in train:', [c.replace('line_', '') for c in line_cols if train[c].sum() > 0])
print('Lines in test: ', [c.replace('line_', '') for c in line_cols if test[c].sum() > 0])

X_train = train[X_COLS]
y_train = train['log_target']
X_test  = test[X_COLS]
y_test  = test['target_ultimate']

## 2. Chain-ladder baseline

The **chain-ladder method** computes median age-to-age (ATA) paid development factors from historical triangles, then chains them to project each snapshot's paid loss to ultimate.

This is the industry-standard reserving method. Any ML improvement is measured against it.

In [ ]:
baseline = ActuarialBaseline().fit(train)
test = test.copy()
test['baseline_pred'] = baseline.predict(test).values

baseline_rmse = np.sqrt(mean_squared_error(y_test, test['baseline_pred']))
print(f'Chain-ladder RMSE: ${baseline_rmse:,.0f}')
print()
print('Fitted ATA factors by development lag:')
print(baseline.get_factors_table().to_string())

## 3. XGBoost model

We train a gradient boosted tree model on the same training set. Key design choices:

- **Log-transformed target** (`log_target`): paid losses are heavy-tailed. Training on log scale stabilises variance and prevents large losses from dominating the loss function. Predictions are exponentiated back to dollars for evaluation.
- **`eval_set`**: monitors validation loss every 50 rounds to catch overfitting in real time.
- **Conservative depth (4)**: deeper trees memorise company-specific noise rather than learning generalizable development patterns.

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, np.log1p(y_test))],
    verbose=50,
)

# Evaluate on dollar scale — log RMSE is not meaningful to actuaries
test['xgb_pred'] = np.expm1(xgb.predict(X_test))
xgb_rmse = np.sqrt(mean_squared_error(y_test, test['xgb_pred']))
improvement = (baseline_rmse - xgb_rmse) / baseline_rmse

print(f'Chain-ladder RMSE: ${baseline_rmse:,.0f}')
print(f'XGBoost RMSE:      ${xgb_rmse:,.0f}')
print(f'Error reduction:   {improvement:.1%}')

## 4. Feature importance

Which signals does the model rely on most? We expect actuarial features like `paid_ratio` and `maturity_pct` to rank highly — if raw `paid_loss` dominates, the model may be doing something trivial.

In [ ]:
feat_imp = (
    pd.Series(xgb.feature_importances_, index=X_COLS)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind='barh', ax=ax, color='#4a90d9')
ax.set_title('XGBoost feature importance', fontsize=13)
ax.set_xlabel('Importance score')
ax.axvline(0, color='#cccccc', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. RMSE by line of business

The aggregate RMSE hides where ML actually helps. We expect the largest improvement in **long-tail lines** (medical malpractice, other liability, workers comp) where development is slower and more volatile.

In [ ]:
test['line'] = pd.from_dummies(test[line_cols]).iloc[:, 0].str.replace('line_', '')

results = []
for line in sorted(test['line'].unique()):
    mask = test['line'] == line
    b_rmse = np.sqrt(mean_squared_error(y_test[mask], test.loc[mask, 'baseline_pred']))
    x_rmse = np.sqrt(mean_squared_error(y_test[mask], test.loc[mask, 'xgb_pred']))
    results.append({'Line': line, 'Chain-ladder': b_rmse, 'XGBoost': x_rmse})

res_df = pd.DataFrame(results).set_index('Line')
print(res_df.map(lambda x: f'${x:,.0f}').to_string())
print()

ax = res_df.plot(kind='bar', figsize=(11, 5), color=['#aaaaaa', '#4a90d9'])
ax.set_title('Reserve estimation error by line of business', fontsize=13)
ax.set_ylabel('RMSE — lower is better')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 6. RMSE by development lag

At **early lags** (1–3), future development is highly uncertain. At **late lags** (8–9), most losses are settled and both methods should converge.

In [ ]:
lag_results = []
for lag in range(1, 10):
    mask = test['dev_lag'] == lag
    b = np.sqrt(mean_squared_error(y_test[mask], test.loc[mask, 'baseline_pred']))
    x = np.sqrt(mean_squared_error(y_test[mask], test.loc[mask, 'xgb_pred']))
    lag_results.append({'dev_lag': lag, 'Chain-ladder': b, 'XGBoost': x})

lag_df = pd.DataFrame(lag_results).set_index('dev_lag')

ax = lag_df.plot(figsize=(10, 5), marker='o', color=['#aaaaaa', '#4a90d9'])
ax.set_title('Reserve estimation error by development lag', fontsize=13)
ax.set_ylabel('RMSE — lower is better')
ax.set_xlabel('Development lag (years)')
ax.set_xticks(range(1, 10))
plt.tight_layout()
plt.show()

## 7. Key findings

*(Fill in after running — use your actual numbers)*

- **Overall:** XGBoost reduced reserve estimation RMSE by X% vs chain-ladder ($X vs $X)
- **Best improvement:** [line] saw the largest gain
- **Chain-ladder held its own on:** [line]
- **Most important features:** [from importance chart]
- **By lag:** ML outperformed most at early lags, converged with chain-ladder at lag 8–9